# Part 1: Tokenization

LLMs don't see text — they see sequences of integers. A tokenizer converts between the two.

This notebook walks through building a character-level tokenizer for Shakespeare. The concepts here connect directly to the model architecture you'll write in Part 2.

## Setup

Install dependencies and download the Shakespeare dataset.

In [ ]:
# Install dependencies
!pip install -q tiktoken

# Download the Shakespeare dataset
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "data/shakespeare.txt"
)
print("Downloaded shakespeare.txt")

## Character-Level Tokenization

We use the simplest possible tokenizer: each unique character gets an ID.

No libraries, no pretrained models — just a dictionary mapping characters to integers and back.

In [ ]:
# Load the dataset and build the vocabulary
text = open("data/shakespeare.txt").read()

chars = sorted(set(text))
vocab_size = len(chars)

print(f"Total characters: {len(text):,}")
print(f"Unique characters (vocab size): {vocab_size}")
print(f"Characters: {repr(''.join(chars))}")

In [ ]:
# Build encode/decode functions
stoi = {c: i for i, c in enumerate(chars)}  # string to int
itos = {i: c for c, i in stoi.items()}       # int to string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

# Test it
print(encode("Hello"))               # something like [20, 43, 50, 50, 53]
print(decode(encode("Hello")))       # "Hello"
print(encode("To be or not to be"))

In [ ]:
import torch

# Tokenize the full dataset
tokens = torch.tensor(encode(text), dtype=torch.long)
print(f"Tokenized dataset shape: {tokens.shape}")
print(f"First 50 tokens: {tokens[:50].tolist()}")
print(f"Decoded back: {repr(decode(tokens[:50].tolist()))}")

## Why Character-Level?

Character-level tokenization works well for small datasets like Shakespeare (~1M characters) because:

- **Tiny vocabulary** (65 tokens) — the model's embedding table and output layer are small
- **Dense statistics** — with only 65² = 4,225 possible bigrams, every bigram appears many times in the data
- **No dependencies** — no external libraries needed

This matters more than you'd expect. GPT-2's tokenizer (BPE, 50,257 tokens) turns Shakespeare into ~338k tokens with ~11,700 unique token types. Most token bigrams appear only once or twice — the data is too sparse for the model to learn sequential patterns. We tested this: with BPE on Shakespeare, training loss gets stuck at ~6.3 (unigram frequency) and never improves. With character-level, it drops to ~1.5.

In [ ]:
# Demonstrate bigram density: how often does each bigram appear?
from collections import Counter

bigrams = Counter(zip(text, text[1:]))
print(f"Total unique bigrams in text: {len(bigrams)}")
print(f"Possible bigrams (vocab²): {vocab_size**2}")
print(f"Bigram coverage: {len(bigrams)/vocab_size**2:.1%}")
print()
print("Top 10 most common bigrams:")
for bigram, count in bigrams.most_common(10):
    print(f"  {repr(''.join(bigram))}: {count:,}")

## The Tradeoff

Character-level doesn't scale to large datasets:

- **Sequences are ~3x longer** than BPE (more compute per sample)
- **The model has to learn spelling from scratch** — it has no concept of "words"
- **It wastes capacity on predictable patterns** — `t-h-e` almost always appears together, but the model must predict each character separately

For large datasets (TinyStories, OpenWebText), you'd switch to **Byte-Pair Encoding (BPE)**.

In [ ]:
# Compare with BPE tokenization (GPT-2)
import tiktoken

enc = tiktoken.get_encoding("gpt2")

sample = text[:1000]  # first 1000 characters

char_tokens = encode(sample)
bpe_tokens = enc.encode(sample)

print("=== First 1000 characters of Shakespeare ===")
print(f"Character-level tokens: {len(char_tokens)}")
print(f"BPE tokens (GPT-2):     {len(bpe_tokens)}")
print(f"Compression ratio:      {len(char_tokens)/len(bpe_tokens):.1f}x longer with char-level")
print()
print(f"Character vocab size: {vocab_size}")
print(f"GPT-2 vocab size:     {enc.n_vocab:,}")

# How many unique BPE tokens appear in Shakespeare?
bpe_all = enc.encode(text)
print(f"\nUnique BPE token types in Shakespeare: {len(set(bpe_all)):,} out of {enc.n_vocab:,}")
print("Most token bigrams would appear <2 times — too sparse to learn from.")

## How Vocab Size Connects to the Model

The vocabulary size directly determines two things in the model architecture:

1. **The embedding table** — `nn.Embedding(vocab_size, n_embd)` maps each token ID to a learned vector.
   - With `vocab_size=65`: 65 × 384 = **24,960 parameters** (tiny)
   - With GPT-2's vocab of 50,257: 50,257 × 384 = **19.3M parameters** — nearly half the model

2. **The output layer** — `nn.Linear(n_embd, vocab_size)` produces a probability over all possible next tokens.
   - With 65 tokens, the model chooses from 65 options
   - With 50,257 tokens, it must spread its predictions across 50,257 classes

In [ ]:
# Visualize the parameter count impact of vocab size
n_embd = 384  # our model's embedding dimension

configs = [
    ("Character-level (Shakespeare)", 65),
    ("Word-level (medium)", 10_000),
    ("BPE - GPT-2", 50_257),
]

print(f"{'Tokenizer':<35} {'Vocab Size':>12} {'Embedding Params':>20}")
print("-" * 70)
for name, vocab in configs:
    params = vocab * n_embd
    print(f"{name:<35} {vocab:>12,} {params:>20,}")

## Key Takeaways

- Tokenization converts text ↔ integer sequences
- We use character-level: each unique character gets an ID (`stoi`/`itos` mappings)
- Vocabulary size must match the dataset — 50k vocab on a tiny dataset means most token patterns are too sparse to learn
- Character-level works for small experiments; BPE is needed for large datasets
- The vocab size directly affects model size (embedding table) and training difficulty (output distribution)

**Next: [Part 2 — The Transformer](02-the-transformer.ipynb)**